In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)


In [3]:
help(load_faq_data)

Help on function load_faq_data in module ingest:

load_faq_data()



In [4]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [5]:
index.search("containerd")

[]

In [6]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I don’t see any FAQ entry about running **Olama** locally.

The closest relevant guidance in the FAQ is that you can run the **course locally** instead of Codespaces if you’re comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module. If you do, you should document your setup and keep it reproducible.

If you meant a specific local setup step for Olama, I’d need that FAQ/context entry to answer it accurately.


In [7]:
################

In [8]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Possibly — it depends on the course’s enrollment rules and whether registration is still open.\n\nIf you want, send me:\n- the course name\n- the school/platform\n- whether it’s live or self-paced\n- any deadline or start date you saw\n\nand I can help you figure out whether you can still join and what to do next.'

In [9]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [10]:
ans = search("docker")

In [11]:
len(ans)


4

In [12]:
###### SEARCH TOOL
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [13]:
type(search_tool)

dict

In [14]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [15]:
t=response
print(t)

Response(id='resp_0cd043081991c4d0006a3566cd1b9c819c879f0f8a01ef0acd', created_at=1781884621.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I discovered it late? enrollment late join join course discovered just discovered the course"}', call_id='call_04mJcDcCEXbZJTm4kwFK95wP', name='search', type='function_call', id='fc_0cd043081991c4d0006a3566cd7a3c819c87f07e62b0e02ed6', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='search', parameters={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query text to look up in the course FAQ.'}}, 'required': ['query'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Search the FAQ database for entries matching the given query.')], top_p=0.

In [16]:
print(response)

Response(id='resp_0cd043081991c4d0006a3566cd1b9c819c879f0f8a01ef0acd', created_at=1781884621.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I discovered it late? enrollment late join join course discovered just discovered the course"}', call_id='call_04mJcDcCEXbZJTm4kwFK95wP', name='search', type='function_call', id='fc_0cd043081991c4d0006a3566cd7a3c819c87f07e62b0e02ed6', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='search', parameters={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query text to look up in the course FAQ.'}}, 'required': ['query'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Search the FAQ database for entries matching the given query.')], top_p=0.

In [17]:
call = response.output[0]

In [18]:
call

ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I discovered it late? enrollment late join join course discovered just discovered the course"}', call_id='call_04mJcDcCEXbZJTm4kwFK95wP', name='search', type='function_call', id='fc_0cd043081991c4d0006a3566cd7a3c819c87f07e62b0e02ed6', namespace=None, status='completed')

In [19]:
import json

args = json.loads(call.arguments)
args

{'query': 'Can I join the course if I discovered it late? enrollment late join join course discovered just discovered the course'}

In [20]:
call.arguments
call.name

'search'

In [21]:
results = search(**args)

In [22]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch

In [23]:
result_json = json.dumps(results, indent=2)
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "How should I start the course and follow the weekly workflow?",
    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

In [24]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [25]:
print(function_call_output)

{'type': 'function_call_output', 'call_id': 'call_04mJcDcCEXbZJTm4kwFK95wP', 'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\\n\\nYou can start whenever you want. The videos and GitHub materials are available, and the dead

In [26]:
messages.append(call)
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I discovered it late? enrollment late join join course discovered just discovered the course"}', call_id='call_04mJcDcCEXbZJTm4kwFK95wP', name='search', type='function_call', id='fc_0cd043081991c4d0006a3566cd7a3c819c87f07e62b0e02ed6', namespace=None, status='completed')]

In [27]:
messages.append(function_call_output)


In [28]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [29]:
print(response.output_text)

Yes — you can still join and start learning.

If you want a certificate, make sure you submit your project while submissions are still open.


In [30]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(782, 32)

In [31]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(usage.input_tokens, usage.output_tokens)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001365


In [32]:
#LOOP

In [33]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [34]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [41]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [42]:
response.output

[ResponseOutputMessage(id='msg_01c4ab2cdc3bfd20006a35671c3974819c9657caadafbc8bf2', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nThe FAQ says that even if you just discovered it, you can start learning and submit homework while the course is still open. If you want a certificate, make sure you submit your project before submissions close.\n\nIf you’d like, I can also explain:\n- how to start the course,\n- how certificates work,\n- or the weekly workflow.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [43]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

ASSISTANT:
Yes — you can still join the course.

The FAQ says that even if you just discovered it, you can start learning and submit homework while the course is still open. If you want a certificate, make sure you submit your project before submissions close.

If you’d like, I can also explain:
- how to start the course,
- how certificates work,
- or the weekly workflow.


In [44]:
###RESET

In [46]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]


it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course discovered course can I join registration late enrollment new student FAQ"}
function_call: search {"query":"course enrollment open join discovered course can I join FAQ"}
iteration #2...
ASSISTANT:
Yes, you can still join. You can start learning and submitting homework as long as the submission form is open.

One important note: if you want a certificate, you need to submit your project while submissions are still being accepted.

If you want, I can also help with whether you need to register, how the homework works, or how certificates are earned.


In [47]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [50]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [51]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late registration FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If your goal is just to learn, you can start anytime. If you want a certificate, you need to finish while the course is running and submit your project before submissions close.

If you’d like, I can also help you figure out how to start the course now or how the certificate process works.


In [53]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"Queen's Gambit chess opening course FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find anything in the course FAQ about “queen gambit,” so it looks like that’s outside the course topics.

If you meant **“Queen’s Gambit”** as a chess term or something else, I can’t answer off-FAQ content here. Is there another course-related question or area you want to explore?


In [54]:
###FRAMEWORKS

In [55]:
!uv add toyaikit


Resolved 127 packages in 2.59s                                       
Prepared 7 packages in 886ms                                             
Installed 7 packages in 28ms                                
 + anthropic==0.111.0
 + docstring-parser==0.18.0
 + genai-prices==0.0.66
 + httpcore2==2.4.0
 + httpx2==2.4.0
 + toyaikit==0.0.11
 + truststore==0.10.4


In [56]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [59]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
agent_tools.get_tools()


[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [60]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [61]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [62]:
agent_tools.get_tools()


[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [63]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [64]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


In [65]:
result.cost

CostInfo(input_cost=Decimal('0.00106125'), output_cost=Decimal('0.001287'), total_cost=Decimal('0.00234825'))

In [66]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [67]:
runner.run()

You: can i use docker?


-> Response received


-> Response received


-> Response received


You: stop


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None), EasyInputMessage(content='can i use docker?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"docker use course docker allowed"}', call_id=